In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

c:\Users\manh\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
df = pd.read_csv("./vn30_from_2018_to_2025.csv")
df = df[["symbol", "time", "close"]]
df

FileNotFoundError: [Errno 2] No such file or directory: './vn30_from_2018_to_2025.csv'

# Add feature

In [ ]:
def add_log_return(df=None, time_col="time"):
    df = df.copy()

    df[time_col] = pd.to_datetime(df[time_col])
    df = df.sort_values(by=['symbol', time_col])

    df['close'] = df['close'].replace(0, np.nan)
    
    df['log_return'] = df.groupby('symbol')['close'].transform(
        lambda x: np.log(x / x.shift(1))
    )

    df = df.dropna(subset=["log_return"])

    return df

In [ ]:
df2 = add_log_return(df)
df2

,symbol,time,close,log_return
1,ACB,2018-01-03,6.79,-0.002941
2,ACB,2018-01-04,6.81,0.002941
3,ACB,2018-01-05,6.81,0.000000
4,ACB,2018-01-08,7.04,0.033216
5,ACB,2018-01-09,7.07,0.004252
...,...,...,...,...
56974,VRE,2025-12-25,32.25,-0.071780
56975,VRE,2025-12-26,32.00,-0.007782
56976,VRE,2025-12-29,33.20,0.036814
56977,VRE,2025-12-30,32.80,-0.012121


# Add label

In [ ]:
def add_target(df, horizon=1):
    df = df.copy()
    df["target"] = df.groupby("symbol")["log_return"].shift(-horizon)
    df['target_time'] = df.groupby('symbol')['time'].shift(-horizon)
    df = df.dropna(subset=["target"])
    return df

In [ ]:
df3 = add_target(df2)
df3

,symbol,time,close,log_return,target,target_time
1,ACB,2018-01-03,6.79,-0.002941,0.002941,2018-01-04
2,ACB,2018-01-04,6.81,0.002941,0.000000,2018-01-05
3,ACB,2018-01-05,6.81,0.000000,0.033216,2018-01-08
4,ACB,2018-01-08,7.04,0.033216,0.004252,2018-01-09
5,ACB,2018-01-09,7.07,0.004252,-0.012812,2018-01-10
...,...,...,...,...,...,...
56973,VRE,2025-12-24,34.65,0.011611,-0.071780,2025-12-25
56974,VRE,2025-12-25,32.25,-0.071780,-0.007782,2025-12-26
56975,VRE,2025-12-26,32.00,-0.007782,0.036814,2025-12-29
56976,VRE,2025-12-29,33.20,0.036814,-0.012121,2025-12-30


In [ ]:
df3.symbol.nunique()

30

In [ ]:
df3.describe()

,time,close,log_return,target,target_time
count,56917,56917.000000,56917.000000,56917.000000,56917
mean,2022-01-27 01:33:56.333608448,35.454759,0.000490,0.000488,2022-01-28 12:39:34.910835200
min,2018-01-03 00:00:00,1.920000,-0.164622,-0.164622,2018-01-04 00:00:00
25%,2020-02-06 00:00:00,12.560000,-0.009163,-0.009174,2020-02-07 00:00:00
50%,2022-02-08 00:00:00,24.560000,0.000000,0.000000,2022-02-09 00:00:00
75%,2024-01-18 00:00:00,51.800000,0.009947,0.009947,2024-01-19 00:00:00
max,2025-12-30 00:00:00,219.100000,0.155079,0.155079,2025-12-31 00:00:00
std,NaN,30.168559,0.021551,0.021550,NaN


# Train/Val/Test

In [ ]:
def split_time_series(df,
                      train_start='2018-01-01',
                      train_end='2023-01-01',
                      val_end='2024-01-01',
                      test_end=None,
                      time_col='time'):
    
    df = df.copy()

    train_df = df[
        (df[time_col] >= train_start) &
        (df[time_col] < train_end)
    ]

    val_df = df[
        (df[time_col] >= train_end) &
        (df[time_col] < val_end)
    ]

    if test_end is not None:
        test_df = df[
            (df[time_col] >= val_end) &
            (df[time_col] < test_end)
        ]
    else:
        test_df = df[df[time_col] >= val_end]

    return train_df, val_df, test_df

In [ ]:
train_df, val_df, test_df = split_time_series(df3)
train_df.shape, val_df.shape, test_df.shape

((35091, 6), (7221, 6), (14605, 6))

In [ ]:
print(train_df['time'].min(), train_df['time'].max())
print(val_df['time'].min(), val_df['time'].max())
print(test_df['time'].min(), test_df['time'].max())

2018-01-03 00:00:00 2022-12-30 00:00:00
2023-01-03 00:00:00 2023-12-29 00:00:00
2024-01-02 00:00:00 2025-12-30 00:00:00


In [ ]:
print(train_df["symbol"].nunique())
print(val_df["symbol"].nunique())
print(test_df["symbol"].nunique())

29
29
30


In [ ]:
set(test_df.symbol.unique()) - set(train_df.symbol.unique())

{'VPL'}

In [ ]:
symbols_in_train = train_df['symbol'].unique()

test_df = test_df[test_df['symbol'].isin(symbols_in_train)]

test_df['symbol'].nunique(), test_df.shape

(29, (14442, 6))

# Create sequences

In [ ]:
def create_sequences(df, window=20, feature_col='log_return', target_col='target'):
    X, y, meta = [], [], [] 
    
    for symbol, sub_df in df.groupby('symbol'):
        sub_df = sub_df.sort_values('time').reset_index(drop=True)

        values_x = sub_df[feature_col].values
        values_y = sub_df[target_col].values
        
        for i in range(len(sub_df) - window):
            X.append(values_x[i : i + window])
            y.append(values_y[i + window - 1])
            meta.append({
                'symbol': symbol,
                'time': sub_df['target_time'].iloc[i + window - 1]
            })
    
    X = np.array(X).reshape(-1, window, 1)
    y = np.array(y)
    meta_df = pd.DataFrame(meta)
    
    return X, y, meta_df

In [ ]:
X_train, y_train, meta_train = create_sequences(train_df)
X_train.shape, y_train.shape, meta_train.shape

((34511, 20, 1), (34511,), (34511, 2))

In [ ]:
X_val, y_val, meta_val = create_sequences(val_df)
X_val.shape, y_val.shape, meta_val.shape

((6641, 20, 1), (6641,), (6641, 2))

In [ ]:
X_test, y_test, meta_test = create_sequences(test_df)
X_test.shape, y_test.shape, meta_test.shape

((13862, 20, 1), (13862,), (13862, 2))

In [ ]:
34511+20*29, 6641+20*29, 13862+20*29

(35091, 7221, 14442)

# Building LSTM -- each stock/model